# Multi-model AoA read-out — BabyLM 2026

Compare checkpoints on how much of children's acquisition order (Wordbank CDI, 619 words) a frozen linear read-out recovers from their representations.

**Scope**: CDI words only (no corpus-AoA training words). Bare + contextual representations, all layers, per model:
1. ridge read-out (5-fold CV) -> Spearman rho vs Wordbank AoA
2. semantic-structure features (cohesion / distinctiveness / entropy) added to Braginsky-4
3. bootstrap CIs + pairwise comparison vs the full-data reference model


In [ ]:
import importlib.metadata
ver = importlib.metadata.version("transformers")
print("transformers", ver)
if not ver.startswith("4."):
    import subprocess
    subprocess.run(["pip", "install", "-q", "transformers==4.51.3"], check=True)
    raise SystemExit("Installed transformers 4.51.3 — now Runtime > Restart session, then rerun from Step 1.")
print("version OK, proceed")

transformers 4.51.3
version OK, proceed


In [ ]:
import os, re, glob, warnings, gc
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except Exception: IN_COLAB = False

SENTINEL = os.path.join("data", "cdi_items_with_aoa.csv")

def find_repo_root(sentinel=SENTINEL):
    """Repo root = the directory containing data/cdi_items_with_aoa.csv.

    Walks up from the working directory first (covers a local clone, where the
    notebook runs from notebooks/), then falls back to the usual Colab and
    Drive locations.
    """
    here = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.exists(os.path.join(here, sentinel)): return here
        parent = os.path.dirname(here)
        if parent == here: break
        here = parent
    for c in ["/content/drive/MyDrive/aoa-pipeline", "/content/drive/MyDrive/aoa_pipeline",
              "/content/aoa-pipeline", "/content/drive/MyDrive", "/content"]:
        if os.path.exists(os.path.join(c, sentinel)): return c
    for r in ["/content/drive/MyDrive", "/content"]:
        h = glob.glob(os.path.join(r, "**", sentinel), recursive=True)
        if h: return os.path.dirname(os.path.dirname(h[0]))
    raise FileNotFoundError(
        f"Could not locate the repo root (no {sentinel} found). "
        "Set BASE by hand, e.g. BASE = '/content/drive/MyDrive/aoa-pipeline'.")

BASE = find_repo_root()          # or hard-set BASE = "/content/drive/MyDrive/aoa-pipeline"
DATA_DIR = os.path.join(BASE, "data")
VEC_DIR  = os.path.join(BASE, "outputs", "vectors")     # large .npz, not tracked in git
RES_DIR  = os.path.join(BASE, "outputs", "results")     # result tables
FIG_DIR  = os.path.join(BASE, "outputs", "figures")     # figures
for _d in (VEC_DIR, RES_DIR, FIG_DIR): os.makedirs(_d, exist_ok=True)
OUT = VEC_DIR                    # kept so `os.path.exists(OUT)`-style checks still read naturally

def outpath(name):
    """Route an output filename to vectors/ (.npz), figures/ (.png) or results/ (everything else)."""
    ext = os.path.splitext(name)[1].lower()
    return os.path.join({".npz": VEC_DIR, ".png": FIG_DIR}.get(ext, RES_DIR), name)

def find_file(name):
    for d in [DATA_DIR, BASE, os.getcwd()]:
        q = os.path.join(d, name)
        if os.path.exists(q): return q
    h = glob.glob(os.path.join(BASE, "**", name), recursive=True)
    if h: return h[0]
    raise FileNotFoundError(name)

NEEDED = ["cdi_items_with_aoa.csv", "contexts_all_childes.csv", "childes_freq_eng-na.csv",
          "brysbaert_concreteness.txt", "perry_2015_babiness.csv", "childes_mlu.csv"]
status = {}
for n in NEEDED:
    try: find_file(n); status[n] = True
    except FileNotFoundError: status[n] = False
print("BASE =", BASE)
print("data =", DATA_DIR, "| outputs =", os.path.join(BASE, "outputs"))
for n, ok in status.items(): print(("  OK   " if ok else "  MISSING ->") + f" {n}")
missing = [n for n, ok in status.items() if not ok]
assert not missing, f"Put these in {DATA_DIR}: {missing}"

CONFIG = dict(max_ctx_reps=50, max_len=40, ridge_lambda=10.0, cv_folds=5, seed=0)
print("CONFIG:", CONFIG)


##Load data (CDI test set, Braginsky predictors, CHILDES contexts)

In [ ]:
def clean_word(x):
    x = str(x).lower().strip(); x = re.sub(r"\s*\(.*?\)", "", x); x = re.sub(r"\*.*$", "", x); return x.strip()

def load_cdi():
    df = pd.read_csv(find_file("cdi_items_with_aoa.csv")); df["key"] = df["word"].map(clean_word)
    df = df[df["key"].str.match(r"^[a-z'\-]+$") & df["key"].ne("")]
    return (df[["key", "category", "lexical_category", "aoa"]]
            .dropna(subset=["aoa"]).drop_duplicates("key").reset_index(drop=True))

def load_braginsky():
    fr = pd.read_csv(find_file("childes_freq_eng-na.csv")); fr["key"] = fr["word"].map(clean_word)
    fr = fr.groupby("key", as_index=False)["childprod_logfreq"].mean().rename(columns={"childprod_logfreq": "logf"})
    co = pd.read_csv(find_file("brysbaert_concreteness.txt"), sep="\t"); co["key"] = co["Word"].map(clean_word)
    co = co.groupby("key", as_index=False)["Conc.M"].mean().rename(columns={"Conc.M": "conc"})
    ba = pd.read_csv(find_file("perry_2015_babiness.csv")); bcol = "babyAVG" if "babyAVG" in ba.columns else "rating"
    ba["key"] = ba["word"].map(clean_word); ba = ba.groupby("key", as_index=False)[bcol].mean().rename(columns={bcol: "baby"})
    ml = pd.read_csv(find_file("childes_mlu.csv")); ml["key"] = ml["word"].map(clean_word)
    ml = ml.groupby("key", as_index=False)["mean_sent_length"].mean().rename(columns={"mean_sent_length": "mlu"})
    return fr.merge(co, on="key", how="outer").merge(ba, on="key", how="outer").merge(ml, on="key", how="outer")

def load_contexts():
    c = pd.read_csv(find_file("contexts_all_childes.csv"))
    return {k: g["sentence"].astype(str).tolist() for k, g in c.groupby("key")}

cdi = load_cdi(); brag = load_braginsky(); contexts = load_contexts()
ALL_WORDS = cdi["key"].tolist()                      # CDI words only — multi-model scope
n_ctx = sum(1 for w in ALL_WORDS if contexts.get(w))
print(f"CDI words: {len(ALL_WORDS)} | with >=1 CHILDES context: {n_ctx}")

CDI words: 619 | with >=1 CHILDES context: 619



Changes vs the single-model pipeline, all needed for cross-tokenizer robustness:
- word located by **character offsets** (`return_offsets_mapping`), not token-subsequence match
- special tokens excluded via `special_tokens_mask`, not `position 0` (GPT-2 has no BOS)
- layer count read from a dummy forward pass, not hardcoded 1-12

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForCausalLM

MODELS = [
    # --- OPT-125M family: data-scale axis, same architecture ---
    dict(id="znhoughton/opt-babylm-125m-20eps-seed964", slug="opt125-babylm150m", trust=False),
    dict(id="babylm/opt-125m-strict-2023",              slug="opt125-babylm100m", trust=False),
    dict(id="babylm/opt-125m-strict-small-2023",        slug="opt125-babylm10m",  trust=False),
    dict(id="facebook/opt-125m",                        slug="opt125-full",       trust=False),
    # --- GPT-2 family: data-scale axis (2025 official baselines) ---
    dict(id="BabyLM-community/babylm-baseline-100m-gpt2", slug="gpt2-babylm100m", trust=False),
    dict(id="BabyLM-community/babylm-baseline-10m-gpt2",  slug="gpt2-babylm10m",  trust=False),
    dict(id="openai-community/gpt2",                      slug="gpt2-full",       trust=False),
    # --- BabyLlama family: 2023 winner + 2024 baselines (distillation) ---
    dict(id="timinar/baby-llama-58m",                   slug="babyllama58-2023",  trust=False),
    dict(id="babylm/babyllama-100m-2024",               slug="babyllama-100m",    trust=False),
    dict(id="babylm/babyllama-10m-2024",                slug="babyllama-10m",     trust=False),
    # --- encoder family: babylm vs full-data ---
    dict(id="babylm/roberta-base-strict-2023",          slug="roberta-babylm100m", trust=False),
    dict(id="FacebookAI/roberta-base",                  slug="roberta-full",       trust=False),
    # --- challenge winners + objective-mix axis (custom code; smoke-gated) ---
    dict(id="lgcharpe/ELC_BERT_baby_100M",              slug="elcbert-100m-2023",  trust=True),
    dict(id="ltg/gpt-bert-babylm-base",                 slug="gptbert-100m-2024",  trust=True),
    dict(id="babylm/ltgbert-100m-2024",                 slug="ltgbert-100m-2024",  trust=True),
    dict(id="BabyLM-community/babylm-baseline-100m-gpt-bert-mixed",        slug="gptbert25-mixed",  trust=True),
    dict(id="BabyLM-community/babylm-baseline-100m-gpt-bert-causal-focus", slug="gptbert25-causal", trust=True),
    dict(id="BabyLM-community/babylm-baseline-100m-gpt-bert-masked-focus", slug="gptbert25-masked", trust=True),
]
REF_SLUG = "opt125-full"       # full-data reference for pairwise comparisons

DEV = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEV)

def load_model(m):
    tok = AutoTokenizer.from_pretrained(m["id"], trust_remote_code=m["trust"])
    if not getattr(tok, "is_fast", False):
        raise RuntimeError(f"{m['id']}: not a fast tokenizer -> no offset mapping; needs a per-model fallback")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if tok.eos_token is not None else tok.unk_token
    assert tok.pad_token is not None, f"{m['id']}: no usable pad token"
    body, last = None, None
    for cls in (AutoModel, AutoModelForMaskedLM, AutoModelForCausalLM):
        try:
            body = cls.from_pretrained(m["id"], trust_remote_code=m["trust"]); break
        except Exception as e:
            last = e
    if body is None:
        raise RuntimeError(f"{m['id']}: all AutoModel classes failed; last error: {last}")
    body = body.to(DEV).eval()
    with torch.no_grad():                              # dummy forward: layer count + hidden_states support
        pr = tok("hello world", return_tensors="pt")
        pr = {k: v.to(DEV) for k, v in pr.items() if k in ("input_ids", "attention_mask")}
        hs = body(**pr, output_hidden_states=True).hidden_states
    assert hs is not None and len(hs) > 1, f"{m['id']}: model does not return hidden_states"
    layers = list(range(1, len(hs)))                   # hs[0] is the embedding layer, skipped
    return tok, body, layers, hs[-1].shape[-1]

def word_span_tokens(offsets, sent, word):
    """Token indices covering `word` in `sent`, via character offsets — tokenizer-agnostic."""
    m = re.search(r"(?<![a-z'])" + re.escape(word) + r"(?![a-z'])", sent, re.IGNORECASE)
    if m is None: return []
    s, e = m.span()
    return [i for i, (a, b) in enumerate(offsets) if a < e and b > s and b > a]

def extract_contextual(tok, body, layers, H, words, contexts, max_ctx=50, max_len=40, bs=64):
    acc = {L: [] for L in layers}; matched = 0
    with torch.no_grad():
        for wi, w in enumerate(words):
            sents = contexts.get(w, [])[:max_ctx]
            per = {L: [] for L in layers}
            if sents:
                enc = tok(sents, padding=True, truncation=True, max_length=max_len,
                          return_tensors="pt", return_offsets_mapping=True)
                offs = enc.pop("offset_mapping")
                for i in range(0, len(sents), bs):
                    ids = enc["input_ids"][i:i+bs].to(DEV)
                    am = enc["attention_mask"][i:i+bs].to(DEV)
                    hs = body(input_ids=ids, attention_mask=am, output_hidden_states=True).hidden_states
                    for bb in range(ids.size(0)):
                        ti = word_span_tokens(offs[i+bb].tolist(), sents[i+bb], w)
                        if not ti: continue
                        for L in layers:
                            per[L].append(hs[L][bb, ti].mean(0).float().cpu().numpy())
            if per[layers[0]]: matched += 1
            for L in layers:
                acc[L].append(np.mean(per[L], axis=0) if per[L] else np.full(H, np.nan))
            if (wi + 1) % 100 == 0: print(f"   contextual {wi+1}/{len(words)}")
    print(f"   words with >=1 matched context: {matched}/{len(words)}")
    return {L: np.stack(acc[L]).astype(np.float16) for L in layers}

def extract_bare(tok, body, layers, words, bs=128):
    acc = {L: [] for L in layers}
    with torch.no_grad():
        for i in range(0, len(words), bs):
            b = words[i:i+bs]
            enc = tok([" " + w for w in b], padding=True, return_tensors="pt",
                      return_special_tokens_mask=True)
            sm = enc.pop("special_tokens_mask")
            ids = enc["input_ids"].to(DEV); am = enc["attention_mask"].to(DEV)
            hs = body(input_ids=ids, attention_mask=am, output_hidden_states=True).hidden_states
            keep = (am * (1 - sm.to(DEV))).unsqueeze(-1).float()   # drop pads AND special tokens
            for L in layers:
                acc[L].append(((hs[L] * keep).sum(1) / keep.sum(1).clamp(min=1)).float().cpu().numpy())
    return {L: np.concatenate(acc[L]).astype(np.float16) for L in layers}

def free(*objs):
    for o in objs: del o
    gc.collect()
    if DEV == "cuda": torch.cuda.empty_cache()

print(f"{len(MODELS)} models configured")

device: cuda
18 models configured


##SMOKE TEST (run before extraction, every model must print OK)

10 words x 10 contexts per model. Checks: model loads, returns `hidden_states`, tokenizer offsets match >=90% of contexts. If a model prints `!!` or `X`, stop and fix before Step 5 — do not extract on a failing model.

In [ ]:
SMOKE = {}
TEST_WORDS = ALL_WORDS[:10]
for m in MODELS:
    if all(os.path.exists(outpath(f"vectors_{m['slug']}_{r}.npz")) for r in ("contextual", "bare")):
        SMOKE[m["slug"]] = True
        print(f"OK  {m['slug']}: already extracted, smoke skipped")
        continue
    try:
        tok, body, layers, H = load_model(m)
    except Exception as e:
        print(f"X   {m['slug']}: LOAD FAILED — {type(e).__name__}: {e}")
        SMOKE[m["slug"]] = False
        continue
    n_match, n_tot = 0, 0
    for w in TEST_WORDS:
        sents = contexts.get(w, [])[:10]
        if not sents: continue
        enc = tok(sents, padding=True, truncation=True, max_length=CONFIG["max_len"],
                  return_tensors="pt", return_offsets_mapping=True)
        for bb, s in enumerate(sents):
            n_tot += 1
            if word_span_tokens(enc["offset_mapping"][bb].tolist(), s, w): n_match += 1
    rate = n_match / max(n_tot, 1)
    SMOKE[m["slug"]] = rate >= 0.9
    print(f"{'OK  ' if rate >= 0.9 else '!!  '}{m['slug']}: layers={len(layers)} hidden={H} "
          f"ctx match {n_match}/{n_tot} ({rate:.0%})")
    free(body, tok)
print("\npassed:", [s for s, v in SMOKE.items() if v])
print("failed:", [s for s, v in SMOKE.items() if not v])

OK  opt125-babylm150m: already extracted, smoke skipped
OK  opt125-babylm100m: already extracted, smoke skipped
OK  opt125-babylm10m: already extracted, smoke skipped
OK  opt125-full: already extracted, smoke skipped
OK  gpt2-babylm100m: already extracted, smoke skipped
OK  gpt2-babylm10m: already extracted, smoke skipped
OK  gpt2-full: already extracted, smoke skipped
OK  babyllama58-2023: already extracted, smoke skipped
OK  babyllama-100m: already extracted, smoke skipped
OK  babyllama-10m: already extracted, smoke skipped
OK  roberta-babylm100m: already extracted, smoke skipped
OK  roberta-full: already extracted, smoke skipped
OK  elcbert-100m-2023: already extracted, smoke skipped
OK  gptbert-100m-2024: already extracted, smoke skipped
OK  ltgbert-100m-2024: already extracted, smoke skipped
OK  gptbert25-mixed: already extracted, smoke skipped
OK  gptbert25-causal: already extracted, smoke skipped
OK  gptbert25-masked: already extracted, smoke skipped

passed: ['opt125-babylm150m

##Full extraction

In [ ]:
for m in MODELS:
    if not SMOKE.get(m["slug"], False):
        print(f"skip {m['slug']} (smoke test not passed)"); continue
    paths = {r: outpath(f"vectors_{m['slug']}_{r}.npz") for r in ("contextual", "bare")}
    if all(os.path.exists(p) for p in paths.values()):
        print(f"skip {m['slug']} (already extracted)"); continue
    print(f"\n===== {m['slug']} ({m['id']}) =====")
    tok, body, layers, H = load_model(m)
    ctxv = extract_contextual(tok, body, layers, H, ALL_WORDS, contexts,
                              max_ctx=CONFIG["max_ctx_reps"], max_len=CONFIG["max_len"])
    bare = extract_bare(tok, body, layers, ALL_WORDS)
    for name, reps in [("contextual", ctxv), ("bare", bare)]:
        arr = np.stack([reps[L] for L in layers])          # (n_layers, n_words, H) float16
        np.savez_compressed(paths[name], reps=arr, words=np.array(ALL_WORDS),
                            layers=np.array(layers), hidden=np.array(H))
    free(body, tok, ctxv, bare)
    print(f"saved vectors_{m['slug']}_*.npz")
print("\nextraction done")

skip opt125-babylm150m (already extracted)
skip opt125-babylm100m (already extracted)
skip opt125-babylm10m (already extracted)
skip opt125-full (already extracted)
skip gpt2-babylm100m (already extracted)
skip gpt2-babylm10m (already extracted)
skip gpt2-full (already extracted)
skip babyllama58-2023 (already extracted)
skip babyllama-100m (already extracted)
skip babyllama-10m (already extracted)
skip roberta-babylm100m (already extracted)
skip roberta-full (already extracted)
skip elcbert-100m-2023 (already extracted)
skip gptbert-100m-2024 (already extracted)
skip ltgbert-100m-2024 (already extracted)
skip gptbert25-mixed (already extracted)
skip gptbert25-causal (already extracted)
skip gptbert25-masked (already extracted)

extraction done


##read-out (ridge, 5-fold CV), every model x rep x layer

Per-fold z-scoring; words with no matched context (NaN vectors) are dropped per model and the count is reported in the CSV. Best layer here is selected on the same CV rho — for the paper's headline number, re-select the layer inside a nested CV (note in Step 9).

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from scipy.stats import spearmanr

def probe_layer(X, y, lam=10.0, folds=5, seed=0):
    ok = ~np.isnan(X).any(1)
    Xo, yo = X[ok], y[ok]
    pred = np.full(len(yo), np.nan)
    for tr, te in KFold(folds, shuffle=True, random_state=seed).split(Xo):
        mu, sd = Xo[tr].mean(0), Xo[tr].std(0) + 1e-8
        r = Ridge(alpha=lam).fit((Xo[tr] - mu) / sd, yo[tr])
        pred[te] = r.predict((Xo[te] - mu) / sd)
    rho = float(spearmanr(pred, yo)[0])
    pred_full = np.full(len(y), np.nan); pred_full[ok] = pred
    return rho, pred_full, ok

y_all = cdi["aoa"].values.astype(float)
res, PRED = [], {}
for m in MODELS:
    for rep in ("contextual", "bare"):
        p = outpath(f"vectors_{m['slug']}_{rep}.npz")
        if not os.path.exists(p): continue
        z = np.load(p)
        assert (z["words"] == np.array(ALL_WORDS)).all(), f"word order mismatch in {p}"
        reps, layers = z["reps"].astype(np.float32), z["layers"].tolist()
        best_rho = -9
        for li, L in enumerate(layers):
            rho, pred_full, ok = probe_layer(reps[li], y_all, lam=CONFIG["ridge_lambda"],
                                             folds=CONFIG["cv_folds"], seed=CONFIG["seed"])
            res.append(dict(checkpoint=m["slug"], rep=rep, layer=int(L),
                            rel_depth=round((li + 1) / len(layers), 3),
                            rho=round(rho, 4), n=int(ok.sum())))
            if rho > best_rho:
                best_rho = rho
                PRED[(m["slug"], rep)] = dict(layer=int(L), rho=rho, pred=pred_full, ok=ok)
        b = PRED[(m["slug"], rep)]
        print(f"{m['slug']:20s} {rep:10s} best layer {b['layer']:2d}  rho={b['rho']:.3f}  (n={int(b['ok'].sum())})")
READOUT = pd.DataFrame(res)
READOUT.to_csv(outpath("readout_all_models.csv"), index=False)
print("\nsaved readout_all_models.csv |", READOUT.shape)

opt125-babylm150m    contextual best layer  7  rho=0.619  (n=618)
opt125-babylm150m    bare       best layer  7  rho=0.565  (n=619)
opt125-babylm100m    contextual best layer  2  rho=0.585  (n=619)
opt125-babylm100m    bare       best layer  1  rho=0.559  (n=619)
opt125-babylm10m     contextual best layer  1  rho=0.475  (n=619)
opt125-babylm10m     bare       best layer  8  rho=0.385  (n=619)
opt125-full          contextual best layer  6  rho=0.584  (n=619)
opt125-full          bare       best layer  5  rho=0.515  (n=619)
gpt2-babylm100m      contextual best layer  1  rho=0.633  (n=619)
gpt2-babylm100m      bare       best layer  1  rho=0.619  (n=619)
gpt2-babylm10m       contextual best layer  4  rho=0.647  (n=619)
gpt2-babylm10m       bare       best layer 10  rho=0.650  (n=619)
gpt2-full            contextual best layer  9  rho=0.486  (n=619)
gpt2-full            bare       best layer  7  rho=0.390  (n=619)
babyllama58-2023     contextual best layer 16  rho=0.510  (n=619)
babyllama5

## Semantic structure (cohesion / distinctiveness / entropy) + Braginsky-4

`semantic_three` and `reg_cv` are verbatim from `run_aoa_pipeline_FINAL.ipynb`, so numbers are directly comparable to the 150M paper. Features are computed within the CDI set, as before. Baselines (freq-only, Braginsky-4) are model-independent and computed once.

In [ ]:
def semantic_three(reps, cats, kbins=20):
    X = np.asarray(reps, float); Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    S = Xn @ Xn.T; np.fill_diagonal(S, np.nan)
    cat = np.asarray(cats); N = len(cat)
    coh = np.full(N, np.nan); dist = np.full(N, np.nan); ent = np.full(N, np.nan)
    for i in range(N):
        same = (cat == cat[i]); same[i] = False; other = (cat != cat[i])
        coh[i] = np.nanmean(S[i, same]) if same.any() else np.nan
        dist[i] = coh[i] - (np.nanmean(S[i, other]) if other.any() else np.nan)
        row = S[i, ~np.isnan(S[i])]
        if len(row):
            h, _ = np.histogram(row, bins=kbins, range=(-1, 1)); pp = h / h.sum()
            ent[i] = -(pp[pp > 0] * np.log(pp[pp > 0])).sum()
    return coh, dist, ent

def spearman(a, b):
    a = pd.Series(np.asarray(a, float)).rank(); b = pd.Series(np.asarray(b, float)).rank()
    return float(np.corrcoef(a, b)[0, 1])

def reg_cv(X, y, cats, k=5, seed=0, zscore=True):
    X = np.asarray(X, float)
    if X.ndim == 1: X = X[:, None]
    if zscore: X = (X - np.nanmean(X, 0)) / (np.nanstd(X, 0) + 1e-9)
    y = np.asarray(y, float)
    idx = np.random.RandomState(seed).permutation(len(y)); fold = np.array_split(idx, k)
    yhat = np.full(len(y), np.nan)
    for f in range(k):
        te = fold[f]; tr = np.concatenate([fold[j] for j in range(k) if j != f])
        A = np.c_[X[tr], np.ones(len(tr))]; c, _, _, _ = np.linalg.lstsq(A, y[tr], rcond=None)
        yhat[te] = np.c_[X[te], np.ones(len(te))] @ c
    err = np.abs(yhat - y); cv = np.asarray(cats)
    ss = 1 - np.sum((yhat - y) ** 2) / np.sum((y - y.mean()) ** 2)
    out = {"MAD_overall": float(np.mean(err)), "R2": float(ss), "spearman": spearman(yhat, y), "n": int(len(y))}
    for cc in pd.unique(cv):
        msk = cv == cc
        out[f"MAD_{cc}"] = float(np.mean(err[msk]))
        out[f"rho_{cc}"] = spearman(yhat[msk], y[msk]) if msk.sum() > 2 else np.nan
    return out

ROWS = []
def emit(model, metrics, checkpoint=None, rep=None, layer=None):
    for kk, v in metrics.items():
        if kk == "n": continue
        if kk == "spearman": cat, met = "all", "spearman"
        elif kk == "R2": cat, met = "all", "R2"
        elif kk == "MAD_overall": cat, met = "all", "MAD_months"
        elif kk.startswith("rho_"): cat, met = kk[4:], "spearman"
        elif kk.startswith("MAD_"): cat, met = kk[4:], "MAD_months"
        else: continue
        ROWS.append({"model": model, "checkpoint": checkpoint, "rep": rep, "layer": layer,
                     "eval_category": cat, "metric": met, "value": v, "n": metrics.get("n")})

d = cdi.merge(brag, on="key", how="left")
mm = d.dropna(subset=["logf"]); emit("freq_only", reg_cv(mm["logf"], mm["aoa"], mm["lexical_category"]))
mm = d.dropna(subset=["logf", "conc", "baby", "mlu"])
emit("braginsky4", reg_cv(mm[["logf", "conc", "baby", "mlu"]], mm["aoa"], mm["lexical_category"]))
print("baselines done")

for m in MODELS:
    for rep in ("contextual", "bare"):
        p = outpath(f"vectors_{m['slug']}_{rep}.npz")
        if not os.path.exists(p): continue
        z = np.load(p); reps, layers = z["reps"].astype(np.float32), z["layers"].tolist()
        for li, L in enumerate(layers):
            coh, dist, ent = semantic_three(reps[li], cdi["category"].values)
            s = pd.DataFrame(dict(key=cdi["key"], aoa=cdi["aoa"], lexical_category=cdi["lexical_category"],
                                  cohesion=coh, distinctiveness=dist, entropy=ent))
            s3 = s.dropna(subset=["cohesion", "distinctiveness", "entropy"])
            emit("semantic", reg_cv(s3[["cohesion", "distinctiveness", "entropy"]], s3["aoa"], s3["lexical_category"]),
                 checkpoint=m["slug"], rep=rep, layer=int(L))
            sb = s.merge(d[["key", "logf", "conc", "baby", "mlu"]], on="key", how="left").dropna(
                subset=["cohesion", "distinctiveness", "entropy", "logf", "conc", "baby", "mlu"])
            emit("braginsky4+semantic",
                 reg_cv(sb[["logf", "conc", "baby", "mlu", "cohesion", "distinctiveness", "entropy"]],
                        sb["aoa"], sb["lexical_category"]),
                 checkpoint=m["slug"], rep=rep, layer=int(L))
    print("semantic done:", m["slug"])

RES = pd.DataFrame(ROWS)
RES.to_csv(outpath("results_multimodel.csv"), index=False)
print("saved results_multimodel.csv |", RES.shape)

baselines done
semantic done: opt125-babylm150m
semantic done: opt125-babylm100m
semantic done: opt125-babylm10m
semantic done: opt125-full
semantic done: gpt2-babylm100m
semantic done: gpt2-babylm10m
semantic done: gpt2-full
semantic done: babyllama58-2023
semantic done: babyllama-100m
semantic done: babyllama-10m
semantic done: roberta-babylm100m
semantic done: roberta-full
semantic done: elcbert-100m-2023
semantic done: gptbert-100m-2024
semantic done: ltgbert-100m-2024
semantic done: gptbert25-mixed
semantic done: gptbert25-causal
semantic done: gptbert25-masked
saved results_multimodel.csv | (10054, 8)


##Main figure: best-layer contextual read-out per model vs Braginsky-4

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

best = (READOUT[READOUT.rep == "contextual"].sort_values("rho", ascending=False)
        .groupby("checkpoint", as_index=False).first())
best = best.sort_values("rho")
b4 = RES[(RES.model == "braginsky4") & (RES.eval_category == "all") &
         (RES.metric == "spearman")]["value"].iloc[0]

fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(best) + 1.5))
ax.barh(best["checkpoint"], best["rho"], color="#4878A8")
for _, r in best.iterrows():
    ax.text(r["rho"] + 0.005, r["checkpoint"], f'{r["rho"]:.3f} (L{int(r["layer"])})', va="center", fontsize=9)
ax.axvline(b4, ls="--", c="crimson", lw=1.2, label=f"Braginsky-4 = {b4:.3f}")
ax.set_xlabel("Spearman rho with Wordbank AoA (best layer, contextual read-out)")
ax.set_xlim(0, max(best["rho"].max(), b4) + 0.12)
ax.legend(loc="lower right"); fig.tight_layout()
fig.savefig(outpath("fig_readout_by_model.png"), dpi=200)
print("saved fig_readout_by_model.png")

saved fig_readout_by_model.png


## Bootstrap CIs and pairwise tests vs the full-data reference

Word-level bootstrap (2000 resamples) on the CV predictions. Pairwise differences are computed on the words valid in **both** models. These CIs treat the layer choice as fixed; for the paper, confirm the headline comparison with nested layer selection.

In [ ]:
RNG = np.random.RandomState(CONFIG["seed"])

def boot_rho_ci(pred_full, y, ok, n=2000):
    idx = np.where(ok & ~np.isnan(pred_full))[0]
    rhos = [float(spearmanr(pred_full[s], y[s])[0])
            for s in (RNG.choice(idx, len(idx), replace=True) for _ in range(n))]
    return np.percentile(rhos, [2.5, 97.5])

rows = []
for (slug, rep), v in PRED.items():
    if rep != "contextual": continue
    lo, hi = boot_rho_ci(v["pred"], y_all, v["ok"])
    rows.append(dict(checkpoint=slug, layer=v["layer"], rho=round(v["rho"], 3),
                     ci_lo=round(lo, 3), ci_hi=round(hi, 3)))
CI = pd.DataFrame(rows).sort_values("rho", ascending=False)
print(CI.to_string(index=False))

if (REF_SLUG, "contextual") in PRED:
    ref = PRED[(REF_SLUG, "contextual")]
    print(f"\npairwise vs {REF_SLUG} (common valid words, delta-rho 95% CI):")
    diff_rows = []
    for (slug, rep), v in PRED.items():
        if rep != "contextual" or slug == REF_SLUG: continue
        common = np.where(v["ok"] & ref["ok"])[0]
        diffs = []
        for _ in range(2000):
            s = RNG.choice(common, len(common), replace=True)
            diffs.append(float(spearmanr(v["pred"][s], y_all[s])[0]) -
                         float(spearmanr(ref["pred"][s], y_all[s])[0]))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        sig = "*" if (lo > 0 or hi < 0) else " "
        diff_rows.append(dict(checkpoint=slug, delta=round(float(np.mean(diffs)), 3),
                              ci_lo=round(lo, 3), ci_hi=round(hi, 3), sig=sig))
        print(f"  {slug:20s} delta={np.mean(diffs):+.3f}  [{lo:+.3f}, {hi:+.3f}] {sig}")
    pd.DataFrame(diff_rows).to_csv(outpath("pairwise_vs_reference.csv"), index=False)
CI.to_csv(outpath("readout_bootstrap_ci.csv"), index=False)
print("\nsaved readout_bootstrap_ci.csv, pairwise_vs_reference.csv")

        checkpoint  layer   rho  ci_lo  ci_hi
    babyllama-100m      1 0.671  0.624  0.710
   gptbert25-mixed      9 0.660  0.612  0.705
  gptbert25-masked     10 0.656  0.606  0.701
    gpt2-babylm10m      4 0.647  0.594  0.694
   gpt2-babylm100m      1 0.633  0.582  0.681
  gptbert25-causal     11 0.628  0.573  0.680
 ltgbert-100m-2024     12 0.623  0.568  0.677
     babyllama-10m      7 0.621  0.562  0.675
 opt125-babylm150m      7 0.619  0.561  0.667
roberta-babylm100m     12 0.599  0.544  0.653
 opt125-babylm100m      2 0.585  0.528  0.642
       opt125-full      6 0.584  0.527  0.638
      roberta-full      2 0.572  0.504  0.628
 elcbert-100m-2023      6 0.570  0.508  0.624
 gptbert-100m-2024     11 0.547  0.487  0.607
  babyllama58-2023     16 0.510  0.442  0.571
         gpt2-full      9 0.486  0.417  0.550
  opt125-babylm10m      1 0.475  0.407  0.539

pairwise vs opt125-full (common valid words, delta-rho 95% CI):
  opt125-babylm150m    delta=+0.035  [-0.025, +0.095]  
  opt

## Nested layer selection (paper-grade headline numbers)

The main-figure rho picks the best layer *post hoc*. Here the layer is chosen inside each outer training fold (inner 5-fold CV), so the reported rho never sees the held-out words during layer choice. Contextual representations only.

In [ ]:
def nested_probe(reps, y, lam=10.0, outer_k=5, inner_k=5, seed=0):
    ok = ~np.isnan(reps[0]).any(1)
    X_all = reps[:, ok, :]; yo = y[ok]; n = len(yo)
    pred = np.full(n, np.nan); chosen = []
    for tr, te in KFold(outer_k, shuffle=True, random_state=seed).split(np.arange(n)):
        best_li, best_r = 0, -9
        for li in range(X_all.shape[0]):
            Xtr = X_all[li][tr]; ytr = yo[tr]
            ip = np.full(len(tr), np.nan)
            for itr, ite in KFold(inner_k, shuffle=True, random_state=seed + 1).split(np.arange(len(tr))):
                mu, sd = Xtr[itr].mean(0), Xtr[itr].std(0) + 1e-8
                r = Ridge(alpha=lam).fit((Xtr[itr] - mu) / sd, ytr[itr])
                ip[ite] = r.predict((Xtr[ite] - mu) / sd)
            rho = float(spearmanr(ip, ytr)[0])
            if rho > best_r: best_r, best_li = rho, li
        chosen.append(best_li)
        mu, sd = X_all[best_li][tr].mean(0), X_all[best_li][tr].std(0) + 1e-8
        r = Ridge(alpha=lam).fit((X_all[best_li][tr] - mu) / sd, yo[tr])
        pred[te] = r.predict((X_all[best_li][te] - mu) / sd)
    pred_full = np.full(len(y), np.nan); pred_full[ok] = pred
    return float(spearmanr(pred, yo)[0]), chosen, pred_full, ok

RNG_N = np.random.RandomState(CONFIG["seed"])
NPRED, nrows = {}, []
for m in MODELS:
    p = outpath(f"vectors_{m['slug']}_contextual.npz")
    if not os.path.exists(p): continue
    z = np.load(p); reps = z["reps"].astype(np.float32); layers = z["layers"].tolist()
    rho, chosen, pred_full, ok = nested_probe(reps, y_all, lam=CONFIG["ridge_lambda"], seed=CONFIG["seed"])
    idx = np.where(ok)[0]
    boots = []
    for _ in range(2000):
        s = RNG_N.choice(idx, len(idx), replace=True)
        boots.append(float(spearmanr(pred_full[s], y_all[s])[0]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    NPRED[m["slug"]] = dict(pred=pred_full, ok=ok)
    nrows.append(dict(checkpoint=m["slug"], rho_nested=round(rho, 4),
                      ci_lo=round(lo, 3), ci_hi=round(hi, 3),
                      layers_chosen=str([layers[c] for c in chosen])))
    print(f"{m['slug']:22s} nested rho={rho:.3f} [{lo:.3f},{hi:.3f}]  layers={[layers[c] for c in chosen]}")
NEST = pd.DataFrame(nrows).sort_values("rho_nested", ascending=False)
NEST.to_csv(outpath("nested_readout.csv"), index=False)

if REF_SLUG in NPRED:
    ref = NPRED[REF_SLUG]
    print(f"\nnested pairwise vs {REF_SLUG}:")
    prows = []
    for slug, v in NPRED.items():
        if slug == REF_SLUG: continue
        common = np.where(v["ok"] & ref["ok"])[0]
        diffs = []
        for _ in range(2000):
            s = RNG_N.choice(common, len(common), replace=True)
            diffs.append(float(spearmanr(v["pred"][s], y_all[s])[0]) -
                         float(spearmanr(ref["pred"][s], y_all[s])[0]))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        sig = "*" if (lo > 0 or hi < 0) else " "
        prows.append(dict(checkpoint=slug, delta=round(float(np.mean(diffs)), 3),
                          ci_lo=round(lo, 3), ci_hi=round(hi, 3), sig=sig))
        print(f"  {slug:22s} delta={np.mean(diffs):+.3f} [{lo:+.3f},{hi:+.3f}] {sig}")
    pd.DataFrame(prows).to_csv(outpath("nested_pairwise_vs_reference.csv"), index=False)
print("\nsaved nested_readout.csv, nested_pairwise_vs_reference.csv")

opt125-babylm150m      nested rho=0.571 [0.509,0.627]  layers=[4, 7, 10, 10, 2]
opt125-babylm100m      nested rho=0.577 [0.518,0.630]  layers=[8, 1, 1, 2, 9]
opt125-babylm10m       nested rho=0.414 [0.346,0.481]  layers=[12, 2, 6, 10, 2]
opt125-full            nested rho=0.560 [0.500,0.615]  layers=[4, 5, 4, 8, 6]
gpt2-babylm100m        nested rho=0.626 [0.572,0.674]  layers=[2, 11, 11, 11, 11]
gpt2-babylm10m         nested rho=0.604 [0.545,0.657]  layers=[10, 5, 4, 10, 7]
gpt2-full              nested rho=0.483 [0.415,0.545]  layers=[9, 9, 9, 10, 10]
babyllama58-2023       nested rho=0.489 [0.423,0.551]  layers=[10, 5, 10, 11, 10]
babyllama-100m         nested rho=0.618 [0.563,0.667]  layers=[12, 8, 16, 3, 14]
babyllama-10m          nested rho=0.571 [0.510,0.628]  layers=[15, 9, 2, 7, 15]
roberta-babylm100m     nested rho=0.583 [0.529,0.638]  layers=[10, 5, 10, 10, 10]
roberta-full           nested rho=0.525 [0.455,0.592]  layers=[10, 10, 12, 10, 7]
elcbert-100m-2023      nested rho=0

## Dimensionality control: PCA to a common 256 dimensions

Hidden sizes differ across models (512-768). PCA (unsupervised, fit on the word vectors) projects every model x layer to 256 components before probing. If the ranking matches the main analysis, hidden-size differences are not driving the result.

In [ ]:
from sklearn.decomposition import PCA

D_COMMON = 256
prow = []
for m in MODELS:
    p = outpath(f"vectors_{m['slug']}_contextual.npz")
    if not os.path.exists(p): continue
    z = np.load(p); reps = z["reps"].astype(np.float32); layers = z["layers"].tolist()
    best = -9
    for li in range(reps.shape[0]):
        X = reps[li]; ok = ~np.isnan(X).any(1)
        Xp = PCA(n_components=D_COMMON, random_state=0).fit_transform(X[ok])
        Xf = np.full((len(X), D_COMMON), np.nan, dtype=np.float32); Xf[ok] = Xp
        rho, _, _ = probe_layer(Xf, y_all, lam=CONFIG["ridge_lambda"],
                                folds=CONFIG["cv_folds"], seed=CONFIG["seed"])
        if rho > best: best, bl = rho, layers[li]
    prow.append(dict(checkpoint=m["slug"], rho_pca=round(best, 4), layer=bl))
    print(f"{m['slug']:22s} PCA-{D_COMMON} best rho={best:.3f} (L{bl})")
PCAC = pd.DataFrame(prow)

main_best = (READOUT[READOUT.rep == "contextual"].sort_values("rho", ascending=False)
             .groupby("checkpoint", as_index=False).first()[["checkpoint", "rho"]]
             .rename(columns={"rho": "rho_main"}))
cmp = main_best.merge(PCAC, on="checkpoint")
rank_r = float(spearmanr(cmp["rho_main"], cmp["rho_pca"])[0])
print(f"\nranking agreement main vs PCA-{D_COMMON}: Spearman = {rank_r:.3f}")
cmp.to_csv(outpath("pca_control.csv"), index=False)
print("saved pca_control.csv")

opt125-babylm150m      PCA-256 best rho=0.618 (L6)
opt125-babylm100m      PCA-256 best rho=0.617 (L3)
opt125-babylm10m       PCA-256 best rho=0.483 (L6)
opt125-full            PCA-256 best rho=0.579 (L4)
gpt2-babylm100m        PCA-256 best rho=0.682 (L3)
gpt2-babylm10m         PCA-256 best rho=0.651 (L6)
gpt2-full              PCA-256 best rho=0.504 (L6)
babyllama58-2023       PCA-256 best rho=0.515 (L9)
babyllama-100m         PCA-256 best rho=0.676 (L3)
babyllama-10m          PCA-256 best rho=0.617 (L16)
roberta-babylm100m     PCA-256 best rho=0.639 (L1)
roberta-full           PCA-256 best rho=0.605 (L11)
elcbert-100m-2023      PCA-256 best rho=0.551 (L12)
gptbert-100m-2024      PCA-256 best rho=0.553 (L11)
ltgbert-100m-2024      PCA-256 best rho=0.586 (L12)
gptbert25-mixed        PCA-256 best rho=0.683 (L7)
gptbert25-causal       PCA-256 best rho=0.680 (L5)
gptbert25-masked       PCA-256 best rho=0.718 (L7)

ranking agreement main vs PCA-256: Spearman = 0.922
saved pca_control.csv


One line per model, relative depth on x. Encoder/hybrid models (RoBERTa, LTG-BERT, ELC-BERT, GPT-BERT) vs pure decoders.

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ENC_KEYS = ("roberta", "ltgbert", "elcbert", "gptbert")
prof = READOUT[READOUT.rep == "contextual"]
fig, ax = plt.subplots(figsize=(8, 5))
for ck, g in prof.groupby("checkpoint"):
    g = g.sort_values("rel_depth")
    enc = any(k in ck for k in ENC_KEYS)
    ax.plot(g["rel_depth"], g["rho"], color=("#B2182B" if enc else "#2166AC"),
            alpha=0.55, lw=1.4)
    b = g.loc[g["rho"].idxmax()]
    ax.scatter([b["rel_depth"]], [b["rho"]],
               color=("#B2182B" if enc else "#2166AC"), s=18, zorder=3)
ax.set_xlabel("relative depth (layer / n_layers)")
ax.set_ylabel("Spearman rho with Wordbank AoA")
ax.legend(handles=[Line2D([0], [0], color="#2166AC", lw=2, label="decoder"),
                   Line2D([0], [0], color="#B2182B", lw=2, label="encoder / hybrid")],
          loc="lower center")
fig.tight_layout()
fig.savefig(outpath("fig_layer_profiles.png"), dpi=200)
print("saved fig_layer_profiles.png")

saved fig_layer_profiles.png
